# HMDA 2025 — SQL layer

The analysis in `01-explore.ipynb` was done in pandas. This notebook reproduces the aggregations in SQL against the same cleaned dataset, loaded into DuckDB.
Results are checked against the pandas output, any mismatch is a bug in the query.

In [44]:
import duckdb
import pandas as pd
from pathlib import Path 

In [4]:
con = duckdb.connect()
con.execute("""
    CREATE VIEW applications AS
    SELECT * FROM read_parquet('../data/processed/applications.parquet')
""")

con.execute("SELECT COUNT(*) AS n FROM applications").df()

,n
0,255970


### Abandonment by channel

Reproduces the headline figure. Should match the pandas result: retail 21.1%,
correspondent 13.6%, broker 12.9%.

In [20]:
con.execute("""
SELECT
    channel,
    COUNT(*) AS applications,
    CAST(SUM(abandoned) AS INTEGER) AS abandoned,
    ROUND(AVG(abandoned)*100, 1) AS abandonment_rate_pct,
    FROM applications
    WHERE channel IS NOT NULL
    GROUP BY channel, 
    ORDER BY abandonment_rate_pct DESC
    
""").df()

,channel,applications,abandoned,abandonment_rate_pct
0,retail,185579,39164,21.1
1,correspondent,17819,2429,13.6
2,broker,41232,5326,12.9


### By channel and loan purpose

Tests whether the channel gap is a composition effect. `HAVING` drops
combinations with fewer than 500 applications, two correspondent cells are too
thin to interpret. Retail is higher in all five purposes.

In [22]:
con.execute("""
    SELECT
        loan_purpose_label,
        channel,
        COUNT(*) AS applications,
        CAST(SUM(abandoned) AS INTEGER) AS abandoned,
        ROUND(100.0 * AVG(abandoned), 1) AS abandonment_rate_pct
    FROM applications
    WHERE channel IS NOT NULL
    GROUP BY loan_purpose_label, channel
    HAVING COUNT(*) >= 500
    ORDER BY loan_purpose_label, abandonment_rate_pct DESC
""").df()

,loan_purpose_label,channel,applications,abandoned,abandonment_rate_pct
0,cash-out refinancing,retail,32567,8880,27.3
1,cash-out refinancing,correspondent,3144,570,18.1
2,cash-out refinancing,broker,5555,911,16.4
3,home improvement,retail,20534,2800,13.6
4,home improvement,broker,931,36,3.9
5,home purchase,retail,75056,14907,19.9
6,home purchase,broker,25091,3273,13.0
7,home purchase,correspondent,9876,1258,12.7
8,other purpose,retail,27676,5182,18.7
9,other purpose,broker,1703,101,5.9


### County-level rates

New analysis with no pandas counterpart. The 500-application floor keeps small
counties from producing extreme rates on tiny samples.

In [26]:
con.execute("""
    SELECT
        county_code,
        COUNT(*) AS applications,
        ROUND(100.0 * AVG(abandoned), 1) AS abandonment_rate_pct
    FROM applications
    WHERE county_code IS NOT NULL
    GROUP BY county_code
    HAVING COUNT(*) >= 500
    ORDER BY abandonment_rate_pct DESC
    LIMIT 10
""").df()


,county_code,applications,abandonment_rate_pct
0,8007.0,881,24.6
1,8051.0,831,24.2
2,8075.0,659,23.8
3,8087.0,1095,23.7
4,8097.0,699,23.7
5,8029.0,1396,23.5
6,8043.0,2229,23.1
7,8037.0,2390,22.9
8,8045.0,2983,22.4
9,8019.0,529,22.3


### Each county against the state

`OVER ()` adds the state-wide rate alongside each county without collapsing the
rows. The state figure is weighted by application volume, not an average of
county rates, the two differ, and only the weighted one is Colorado's actual
abandonment rate.

In [34]:
con.execute("""
    SELECT
        county_code,
        COUNT(*) AS applications,
        ROUND(100.0 * AVG(abandoned), 1) AS county_rate_pct,
        ROUND(100.0 * SUM(SUM(abandoned)) OVER () / SUM(COUNT(*)) OVER (), 1) AS state_rate_pct
    FROM applications
    WHERE county_code IS NOT NULL
    GROUP BY county_code
    HAVING COUNT(*) >= 500
    ORDER BY county_rate_pct DESC
    LIMIT 10
""").df()

,county_code,applications,county_rate_pct,state_rate_pct
0,8007.0,881,24.6,20.3
1,8051.0,831,24.2,20.3
2,8075.0,659,23.8,20.3
3,8087.0,1095,23.7,20.3
4,8097.0,699,23.7,20.3
5,8029.0,1396,23.5,20.3
6,8043.0,2229,23.1,20.3
7,8037.0,2390,22.9,20.3
8,8045.0,2983,22.4,20.3
9,8019.0,529,22.3,20.3


### Abandonment against broker share

Joins county abandonment rates to the share of each county's applications that
came through a broker.

In [40]:
con.execute("""
    WITH county_rates AS (
        SELECT
            county_code,
            COUNT(*) AS applications,
            ROUND(100.0 * AVG(abandoned), 1) AS abandonment_rate_pct
        FROM applications
        WHERE county_code IS NOT NULL
        GROUP BY county_code
        HAVING COUNT(*) >= 500
    ),

    county_channel AS (
        SELECT
            county_code,
            ROUND(100.0 * AVG(CASE WHEN channel = 'broker' THEN 1 ELSE 0 END), 1) AS broker_share_pct
        FROM applications
        WHERE county_code IS NOT NULL
          AND channel IS NOT NULL
        GROUP BY county_code
    )

    SELECT
        r.county_code,
        r.applications,
        r.abandonment_rate_pct,
        c.broker_share_pct
    FROM county_rates AS r
    JOIN county_channel AS c ON r.county_code = c.county_code
    ORDER BY r.abandonment_rate_pct DESC
""").df()

,county_code,applications,abandonment_rate_pct,broker_share_pct
0,8007.0,881,24.6,13.3
1,8051.0,831,24.2,17.8
2,8075.0,659,23.8,11.1
3,8087.0,1095,23.7,18.0
4,8097.0,699,23.7,7.6
5,8029.0,1396,23.5,11.4
6,8043.0,2229,23.1,15.2
7,8037.0,2390,22.9,13.4
8,8045.0,2983,22.4,17.1
9,8019.0,529,22.3,23.5


### Correlation

r = −0.222 across 34 counties: broker-heavy counties abandon slightly less,
directionally consistent with the individual-level result. Too weak to be
distinguishable from zero at this sample size, and county-level patterns do not
establish individual-level ones in any case.

In [42]:
con.execute("""
    WITH county_rates AS (
        SELECT
            county_code,
            COUNT(*) AS applications,
            100.0 * AVG(abandoned) AS abandonment_rate_pct
        FROM applications
        WHERE county_code IS NOT NULL
        GROUP BY county_code
        HAVING COUNT(*) >= 500
    ),

    county_channel AS (
        SELECT
            county_code,
            100.0 * AVG(CASE WHEN channel = 'broker' THEN 1 ELSE 0 END) AS broker_share_pct
        FROM applications
        WHERE county_code IS NOT NULL
          AND channel IS NOT NULL
        GROUP BY county_code
    )

    SELECT
        COUNT(*) AS counties,
        ROUND(CORR(r.abandonment_rate_pct, c.broker_share_pct), 3) AS correlation
    FROM county_rates AS r
    JOIN county_channel AS c ON r.county_code = c.county_code
""").df()

,counties,correlation
0,34,-0.222


### Export for the dashboard

Writes three aggregated tables for Tableau. County codes are zero-padded to
five-character FIPS strings so they match map geometry.

In [46]:
OUT = Path("../data/processed")

exports = {
    "channel_rates.csv": """
        SELECT
            channel,
            COUNT(*) AS applications,
            CAST(SUM(abandoned) AS INTEGER) AS abandoned,
            ROUND(100.0 * AVG(abandoned), 1) AS abandonment_rate_pct
        FROM applications
        WHERE channel IS NOT NULL
        GROUP BY channel
        ORDER BY abandonment_rate_pct DESC
    """,

    "channel_by_purpose.csv": """
        SELECT
            loan_purpose_label,
            channel,
            COUNT(*) AS applications,
            ROUND(100.0 * AVG(abandoned), 1) AS abandonment_rate_pct
        FROM applications
        WHERE channel IS NOT NULL
        GROUP BY loan_purpose_label, channel
        HAVING COUNT(*) >= 500
        ORDER BY loan_purpose_label, abandonment_rate_pct DESC
    """,

    "county_rates.csv": """
        SELECT
            LPAD(CAST(CAST(county_code AS INTEGER) AS VARCHAR), 5, '0') AS county_fips,
            COUNT(*) AS applications,
            ROUND(100.0 * AVG(abandoned), 1) AS abandonment_rate_pct
        FROM applications
        WHERE county_code IS NOT NULL
        GROUP BY county_fips
        HAVING COUNT(*) >= 500
        ORDER BY abandonment_rate_pct DESC
    """,
}

for filename, query in exports.items():
    table = con.execute(query).df()
    table.to_csv(OUT / filename, index=False)
    print(f"{filename:28s} {len(table):>4} rows")

channel_rates.csv               3 rows
channel_by_purpose.csv         13 rows
county_rates.csv               34 rows
